# ViGoEmotions — Ba baseline trên Colab GPU
**Runtime → Change runtime type → T4 GPU**. Upload gói `vigoemotions_colab.zip` tạo bằng `python scripts/package_colab.py --include-results`, sau đó upload riêng ba CSV có quyền sử dụng.

Chạy từ trên xuống trong runtime mới. Mỗi bước vừa in output vừa ghi `reports/logs/`; neural lưu loss và metric mỗi epoch, chọn checkpoint bằng validation Macro-F1 tại 0,5. Nếu một cell lỗi, dừng và đọc log trước khi chạy tiếp để tránh trộn kết quả cũ trong ZIP với lượt mới chưa hoàn tất.

Cuối notebook tải **hai ZIP**: `vigoemotions_results.zip` (bảng, hình, log, xác suất, report) và `vigoemotions_baseline_private.zip` (checkpoint, vocabulary và cấu hình, giữ riêng tư). Lưu notebook vào Drive để giữ output hiển thị. Chưa thực hiện lượt huấn luyện GPU mới trong lần cập nhật notebook này; test không dùng cho model selection.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, os
uploaded = files.upload()  # Chọn vigoemotions_colab.zip
archive = next(name for name in uploaded if name.endswith('.zip'))
ROOT = Path('/content/ViGoEmotions')
ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for item in z.infolist():
        target = (ROOT/item.filename).resolve()
        if not target.is_relative_to(ROOT.resolve()):
            raise ValueError('Unsafe ZIP path')
    z.extractall(ROOT)
os.chdir(ROOT)

In [ ]:
%pip install -q -r requirements-colab.txt
!apt-get -qq update
!apt-get -qq install -y default-jre-headless fonts-dejavu-core

In [ ]:
import torch, subprocess
assert torch.cuda.is_available(), 'Hãy bật GPU runtime trước khi chạy tiếp.'
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run(['java','-version'], check=True)

## Nạp dữ liệu được cấp phép
Chọn đủ `train.csv`, `val.csv`, `test.csv`. Dữ liệu nằm trong runtime riêng và không đưa vào gói kết quả tải xuống.

In [ ]:
uploaded = files.upload()
Path('data').mkdir(exist_ok=True)
for name in ('train.csv','val.csv','test.csv'):
    if name not in uploaded:
        raise FileNotFoundError(f'Thiếu {name}')
    Path('data',name).write_bytes(uploaded[name])
del uploaded

## Ghi log và hiển thị kết quả
Chạy ô định nghĩa helper trước các ô huấn luyện. Log lưu stdout/stderr của kiểm thử, EDA, TF-IDF, BiGRU, PhoBERT và build report. Mỗi file log ghi đè khi chạy lại cùng bước; tải kết quả trước nếu cần giữ lượt cũ. Cell cài môi trường/VnCoreNLP vẫn hiển thị output trực tiếp trong notebook.

In [ ]:
import sys, subprocess, json, os
from datetime import datetime, timezone
import pandas as pd
from IPython.display import display

sys.path.insert(0, str(ROOT/'scripts'))

def run_logged(arguments, name):
    """Stream subprocess output to the notebook and persist it, including failures."""
    log_path = ROOT/'reports/logs'/f'{name}.log'
    log_path.parent.mkdir(parents=True, exist_ok=True)
    env = dict(os.environ, PYTHONUNBUFFERED='1')
    with log_path.open('w', encoding='utf-8') as log:
        header = f"Started UTC: {datetime.now(timezone.utc).isoformat()}\nCommand: {arguments!r}\n"
        log.write(header); log.flush()
        print(header, end='', flush=True)
        with subprocess.Popen(arguments, cwd=ROOT, env=env, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, encoding='utf-8',
                              errors='replace', bufsize=1) as process:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line); log.flush()
            code = process.wait()
        log.write(f'\nExit code: {code}\n')
    print('Log:', log_path)
    if code:
        raise subprocess.CalledProcessError(code, arguments)

def show_training_summary(name):
    run = json.loads((ROOT/'reports'/f'{name}_run.json').read_text())
    history = pd.DataFrame(run['history'])
    history['selected_checkpoint'] = history.epoch.eq(run['best_epoch'])
    history.to_csv(ROOT/'reports/tables'/f'{name}_history.csv', index=False)
    print(f"{name}: GPU={run['device']}; best epoch={run['best_epoch']}; "
          f"training={run['train_seconds']:.2f}s")
    display(pd.DataFrame([run['config'][name]]))
    display(history)


In [ ]:
run_logged([sys.executable,'-m','unittest','discover','-s','tests','-v'], 'tests')
run_logged([sys.executable,'scripts/analyze_data.py'], 'eda')
run_logged([sys.executable,'scripts/train_tfidf.py'], 'tfidf_lr')
display(pd.read_csv('reports/tables/tfidf_lr_metrics.csv'))

## 1. Recurrent baseline: BiGRU
Embedding 128, hidden 128 mỗi hướng, dropout 0.3, 10 epoch, batch 64. Chọn epoch theo validation Macro-F1 tại ngưỡng 0.5; không early-stop bằng test.

In [ ]:
run_logged([sys.executable,'scripts/train_neural.py','--model','gru'], 'gru')
show_training_summary('gru')

## 2. PhoBERT
Checkpoint/revision được khóa trong configs/midterm.json. Dùng VnCoreNLP để word segmentation trước BPE theo model card. Classification head 27 logits, BCEWithLogitsLoss, sigmoid khi suy luận. Lần chạy đầu tải checkpoint và tài nguyên segmenter từ mạng.

In [ ]:
import py_vncorenlp
segmenter_dir = (ROOT/'data/vncorenlp').resolve()
segmenter_dir.mkdir(parents=True,exist_ok=True)
py_vncorenlp.download_model(save_dir=str(segmenter_dir))
run_logged([sys.executable,'scripts/train_neural.py','--model','phobert',
            '--vncorenlp-dir',str(segmenter_dir)], 'phobert')
show_training_summary('phobert')

## 3. So sánh, đường học và xuất báo cáo
Bảng chung gồm Macro/Micro-F1, Hamming Loss, LRAP, tham số và thời gian. Biểu đồ lưu train BCE loss và validation Macro-F1 theo epoch; chưa có validation loss. Đường học không tự chứng minh overfitting/hội tụ. Mỗi lượt chọn checkpoint tốt nhất, không mặc định lấy epoch cuối.

Ca lỗi đã được xuất tự động vào `reports/tables/*_error_cases_val.csv`; nhóm vẫn cần đọc và viết diễn giải thủ công. Các bảng trong phần này dùng validation tại ngưỡng 0,5.

In [ ]:
run_logged([sys.executable,'scripts/build_report.py'], 'build_report')
display(pd.read_csv('reports/tables/model_comparison.csv'))
display(pd.read_csv('reports/tables/neural_style_comparison.csv'))

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name in ('gru', 'phobert'):
    history = pd.read_csv(f'reports/tables/{name}_history.csv')
    axes[0].plot(history.epoch, history.loss, 'o-', label=name)
    axes[1].plot(history.epoch, history.macro_f1, 'o-', label=name)
for axis, title in zip(axes, ('Train BCE loss', 'Validation Macro-F1 at 0.5')):
    axis.set(title=title, xlabel='Epoch')
    axis.legend()
fig.tight_layout()
fig.savefig('reports/figures/neural_learning_curves.png', dpi=150)
plt.show()

In [ ]:
import zipfile
with zipfile.ZipFile('/content/vigoemotions_results.zip','w',zipfile.ZIP_DEFLATED) as z:
    for folder in ('reports','splits'):
        for p in Path(folder).rglob('*'):
            if p.is_file():
                z.write(p,str(p))
files.download('/content/vigoemotions_results.zip')

## 4. Tải checkpoint và vocabulary riêng tư
Chạy cả ô tạo ZIP kết quả ở trên và ô dưới. ZIP kết quả chứa log/history/metrics/probabilities/report nhưng không có trọng số. ZIP riêng tư chứa hai checkpoint tốt nhất, vocabulary BiGRU và config/run metadata để khôi phục.

File trọng số lớn nên đợi tải hoàn tất trước khi đóng runtime. Lưu ZIP riêng tư ở máy/Drive riêng; không đưa vào repo công khai. Các file chỉ lưu `state_dict`, không có optimizer state để resume chính xác giữa chừng.

Sau khi nhập kết quả 04 về project local, tạo lại ZIP code với `--include-results`, rồi chạy notebook 05 để tuning learning rate. Tài liệu: [model card PhoBERT](https://huggingface.co/vinai/phobert-base-v2); hướng dẫn project nằm trong `docs/phobert_tuning.md`.

In [ ]:
private_paths = [
    Path('data/checkpoints/gru_best.pt'),
    Path('data/checkpoints/phobert_best.pt'),
    Path('data/derived/gru_vocab.json'),
    Path('configs/midterm.json'),
    Path('reports/gru_run.json'),
    Path('reports/phobert_run.json'),
]
for path in private_paths:
    if not path.is_file():
        raise FileNotFoundError(f'Thiếu file cần lưu: {path}')
private_zip = Path('/content/vigoemotions_baseline_private.zip')
with zipfile.ZipFile(private_zip, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in private_paths:
        archive.write(path, str(path))
print('Private ZIP:', private_zip, f'({private_zip.stat().st_size / 1024**2:.1f} MiB)')
files.download(str(private_zip))